In [1]:
import pandas as pd

In [2]:
random_state = 2020
embedding_dims = 32
batch_size = 32
epochs = 1
train_test_split = 0.05
dataset_training_frac = 0.25
agency_transactions_cuttoff = 30

In [3]:
df = pd.read_parquet("./data/05-dataframe-with-vectors.parquet") # this takes 4GB

In [4]:
df.head()

,cohort,agency_number,agency_name,holder_last_name,holder_first_initial,amount,vendor,mcc,parsed_amount,transaction_date,...,mcc_vector_758,mcc_vector_759,mcc_vector_760,mcc_vector_761,mcc_vector_762,mcc_vector_763,mcc_vector_764,mcc_vector_765,mcc_vector_766,mcc_vector_767
0,201307,1000,OKLAHOMA STATE UNIVERSITY,Mason,C,890,NACAS,CHARITABLE AND SOCIAL SERVICE ORGANIZATIONS,6.791222,2013-07-30,...,-0.056266,-0.023391,0.001095,0.000461,-0.045571,-0.084020,0.026713,-0.039040,0.049538,0.028440
1,201307,1000,OKLAHOMA STATE UNIVERSITY,Mason,C,368.96,SHERATON HOTEL,SHERATON,5.910688,2013-07-30,...,-0.056678,-0.018424,0.048299,-0.018635,-0.012493,0.027929,-0.023522,-0.031778,-0.024050,-0.006366
2,201307,1000,OKLAHOMA STATE UNIVERSITY,Massey,J,165.82,SEARS.COM 9300,DIRCT MARKETING/DIRCT MARKETERS--NOT ELSEWHERE...,5.110903,2013-07-29,...,-0.009754,-0.014711,-0.006504,0.014008,-0.052526,-0.012174,-0.065906,-0.004480,-0.010071,-0.016485
3,201307,1000,OKLAHOMA STATE UNIVERSITY,Massey,T,96.39,WAL-MART #0137,"GROCERY STORES,AND SUPERMARKETS",4.568402,2013-07-30,...,-0.013979,-0.010851,-0.012691,-0.017168,-0.013498,-0.030440,-0.028582,-0.016572,-0.021503,-0.012944
4,201307,1000,OKLAHOMA STATE UNIVERSITY,Mauro-Herrera,M,125.96,STAPLES DIRECT,"STATIONERY, OFFICE SUPPLIES, PRINTING AND WRIT...",4.835964,2013-07-30,...,-0.020916,-0.053247,0.054345,-0.017741,-0.044836,0.014481,0.016831,0.006052,-0.055828,-0.026703


In [5]:
df = df.sample(frac=dataset_training_frac, random_state=random_state)

In [6]:
drop_cols = ['cohort', 'agency_name', 'holder_last_name', 'holder_first_initial', 'amount', 'vendor', 'mcc', 'transaction_date', 'posted_date'] # keep 'agency_number', 'parsed_amount' and repr of mcc and vendor
label_col = 'agency_number'

In [7]:
df.drop(drop_cols, axis=1, inplace=True)

In [8]:
training_cols = list(set(df.columns).difference(label_col))

In [9]:
# adapted from: https://www.kaggle.com/code/hirotaka0122/triplet-loss-with-pytorch

In [10]:
import torch
import numpy as np
import random
from sklearn import model_selection
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm

In [11]:
X_train, X_test = model_selection.train_test_split(df, test_size=train_test_split, random_state=42)

In [12]:
del df

In [13]:
torch.manual_seed(random_state)
np.random.seed(random_state)
random.seed(random_state)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    torch.cuda.get_device_name()

In [14]:
device

device(type='cuda')

In [15]:
X_test.head()

,agency_number,parsed_amount,vendor_vector_0,vendor_vector_1,vendor_vector_2,vendor_vector_3,vendor_vector_4,vendor_vector_5,vendor_vector_6,vendor_vector_7,...,mcc_vector_758,mcc_vector_759,mcc_vector_760,mcc_vector_761,mcc_vector_762,mcc_vector_763,mcc_vector_764,mcc_vector_765,mcc_vector_766,mcc_vector_767
185481,76000,2.995732,0.012147,0.021508,-0.007825,-0.007432,-0.002976,0.015324,0.052413,0.011674,...,-0.029486,0.017108,0.060356,-0.012803,-0.013699,0.037252,-0.040759,0.004266,-0.037517,-0.013696
227820,77000,2.994732,0.012147,0.021508,-0.007825,-0.007432,-0.002976,0.015324,0.052413,0.011674,...,-0.029486,0.017108,0.060356,-0.012803,-0.013699,0.037252,-0.040759,0.004266,-0.037517,-0.013696
256068,1000,3.248046,-0.008151,-0.019354,-0.016596,-0.003996,-0.020200,0.006140,0.071792,0.026341,...,-0.026932,-0.012795,-0.025433,0.025997,-0.024015,0.006686,-0.045932,-0.024327,-0.068349,0.001787
207528,77000,5.075174,0.025417,-0.045749,-0.009834,-0.022428,-0.030168,0.031007,0.075347,0.024127,...,-0.037274,-0.033133,0.022743,0.024705,-0.026031,-0.039894,-0.025798,-0.048192,-0.041052,-0.046207
361550,80500,4.865378,0.018436,-0.008734,0.022266,-0.032007,0.006713,0.023083,0.054675,-0.001589,...,-0.056678,-0.018424,0.048299,-0.018635,-0.012493,0.027929,-0.023522,-0.031778,-0.024050,-0.006366


In [16]:
X_train.head()

,agency_number,parsed_amount,vendor_vector_0,vendor_vector_1,vendor_vector_2,vendor_vector_3,vendor_vector_4,vendor_vector_5,vendor_vector_6,vendor_vector_7,...,mcc_vector_758,mcc_vector_759,mcc_vector_760,mcc_vector_761,mcc_vector_762,mcc_vector_763,mcc_vector_764,mcc_vector_765,mcc_vector_766,mcc_vector_767
276988,1000,5.193679,0.012147,0.021508,-0.007825,-0.007432,-0.002976,0.015324,0.052413,0.011674,...,-0.030737,0.006508,0.027860,0.017615,0.010312,-0.010796,-0.041277,-0.017827,-0.071669,-0.017251
419100,76000,4.785824,0.012147,0.021508,-0.007825,-0.007432,-0.002976,0.015324,0.052413,0.011674,...,-0.000081,-0.027545,0.055580,-0.041629,-0.030363,0.009376,-0.048923,-0.040697,-0.043675,-0.009006
370667,1000,4.575329,0.012147,0.021508,-0.007825,-0.007432,-0.002976,0.015324,0.052413,0.011674,...,-0.055096,-0.002274,0.033656,-0.002036,-0.036878,0.034396,-0.055493,-0.020433,-0.062871,-0.041654
323386,75000,8.495289,0.012147,0.021508,-0.007825,-0.007432,-0.002976,0.015324,0.052413,0.011674,...,0.003073,-0.002374,0.028515,0.016387,-0.000060,0.044012,-0.040820,-0.041668,-0.027988,-0.018353
16584,1000,5.041617,0.024678,0.000843,-0.012379,0.034011,-0.053550,-0.001318,0.087726,-0.007805,...,0.001332,0.007180,-0.016285,-0.023332,-0.061045,-0.057722,0.047676,0.010300,0.021624,-0.047013


In [17]:
input_dim = len(X_train.columns) - 1

In [18]:
class TransactionsDataset(Dataset):
    def __init__(self, df, training_cols, train=True, agency_transactions_cuttoff=30):
        agency_number_transaction_counts = df.groupby('agency_number').count()['parsed_amount']
        excluded_agencies = agency_number_transaction_counts[agency_number_transaction_counts<agency_transactions_cuttoff]
        self.transactions = df[~df['agency_number'].isin(excluded_agencies.index)]

        self.is_train = train
        self.training_cols = training_cols
        self.agency_numbers = set(self.transactions['agency_number'].unique())

        self.groups = self.transactions.groupby('agency_number')

    def select_negative_agency(self, agency):
        return np.random.choice(list(self.agency_numbers.difference(set([agency]))))

    def select_from_agency(self, agency, exclude_idx=None):
        selected = self.groups.apply(lambda g: g.sample(1))

        if exclude_idx is not None and selected.index[0] == exclude_idx:
            return self.select_from_agency(agency, exclude_idx)
        else:
            return selected
        
    def __len__(self):
        return len(self.transactions)
    
    def __getitem__(self, item):
        anchor_transaction = self.transactions.iloc[item]
        # print('anchor', anchor_transaction.head())
        
        if self.is_train:
            anchor_label = anchor_transaction['agency_number']

            anchor_transaction = anchor_transaction.drop('agency_number', axis='rows').values.astype('float32')

            positive_transaction = self.select_from_agency(anchor_label).drop('agency_number', axis='columns').values[0].astype('float32')            

            negative_transaction = self.select_from_agency(self.select_negative_agency(anchor_label)).drop('agency_number', axis='columns').values[0].astype('float32')
            
            return anchor_transaction, positive_transaction, negative_transaction, anchor_label        
        else:
            return anchor_transaction.drop('agency_number', axis='rows').values.astype('float32'), 0, 0, anchor_transaction['agency_number']

In [19]:
train_ds = TransactionsDataset(X_train, training_cols, train=True, agency_transactions_cuttoff=agency_transactions_cuttoff)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4)

In [20]:
test_ds = TransactionsDataset(X_test, training_cols, train=False, agency_transactions_cuttoff=agency_transactions_cuttoff)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=4)

In [21]:
%time x, y, z, w = train_ds.__getitem__(200)

CPU times: user 484 ms, sys: 318 ms, total: 802 ms
Wall time: 799 ms


In [22]:
x.shape, y.shape, z.shape

((1537,), (1537,), (1537,))

In [23]:
class TripletLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(TripletLoss, self).__init__()
        self.margin = margin
        
    def calc_euclidean(self, x1, x2):
        return (x1 - x2).pow(2).sum(1)
    
    def forward(self, anchor: torch.Tensor, positive: torch.Tensor, negative: torch.Tensor) -> torch.Tensor:
        distance_positive = self.calc_euclidean(anchor, positive)
        distance_negative = self.calc_euclidean(anchor, negative)
        losses = torch.relu(distance_positive - distance_negative + self.margin)

        return losses.mean()

In [24]:
class Network(nn.Module):
    def __init__(self, input_dim=1537, emb_dim=32):
        super(Network, self).__init__()

        self.fc = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.PReLU(),
            nn.Linear(512, 256),
            nn.PReLU(),
            nn.Linear(256, 128),
            nn.PReLU(),
            nn.Linear(128, emb_dim)
        )
        
    def forward(self, x):
        x = self.fc(x)
        return x

In [25]:
model = Network(input_dim, embedding_dims)
model = torch.jit.script(model).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = torch.jit.script(TripletLoss())

In [26]:
model.train()
for epoch in tqdm(range(epochs), desc="Epochs"):
    running_loss = []
    for step, (anchor_transaction, positive_transaction, negative_transaction, anchor_label) in enumerate(tqdm(train_loader, desc="Training", leave=False)):
        anchor = anchor_transaction.to(device)
        positive = positive_transaction.to(device)
        negative = negative_transaction.to(device)
        
        optimizer.zero_grad()

        anchor_out = model(anchor)
        positive_out = model(positive)
        negative_out = model(negative)
        
        loss = criterion(anchor_out, positive_out, negative_out)
        loss.backward()
        optimizer.step()
        
        running_loss.append(loss.cpu().detach().numpy())
    print("Epoch: {}/{} - Loss: {:.4f}".format(epoch+1, epochs, np.mean(running_loss)))

Epochs:   0%|          | 0/1 [00:00<?, ?it/s]

Training:   0%|          | 0/3240 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), "./data/triplet_loss_model.pth")

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
train_results = []
labels = []

model.eval()
with torch.no_grad():
    for transaction, _, _, label in tqdm(test_loader):
        train_results.append(model(transaction.to(device)).cpu().numpy())
        labels.append(label)
        
train_results = np.concatenate(train_results)
labels = np.concatenate(labels)
train_results.shape

In [ ]:
train_results.shape

In [ ]:
plt.figure(figsize=(15, 10), facecolor="azure")
for label in np.unique(labels):
    tmp = train_results[labels==label]
    plt.scatter(tmp[:, 0], tmp[:, 1], label=label)

plt.legend()
plt.show()